In [ ]:
import os
import torch
import argparse
import torchvision


from diffusers.schedulers import (DDIMScheduler, DDPMScheduler, PNDMScheduler, 
                                  EulerDiscreteScheduler, DPMSolverMultistepScheduler, 
                                  HeunDiscreteScheduler, EulerAncestralDiscreteScheduler,
                                  DEISMultistepScheduler, KDPM2AncestralDiscreteScheduler)
from diffusers.schedulers.scheduling_dpmsolver_singlestep import DPMSolverSinglestepScheduler
from diffusers.models import AutoencoderKL, AutoencoderKLTemporalDecoder
from omegaconf import OmegaConf
from transformers import T5EncoderModel, T5Tokenizer

import os, sys
sys.path.append(os.path.split(sys.path[0])[0])
from pipeline_latte_watermark import LattePipelineWatermark
from models import get_models
from utils import save_video_grid
import imageio
from torchvision.utils import save_image
from collections import OrderedDict

from customization import customize_vae_decoder
from torchvision.models import resnet50, ResNet50_Weights
from attribution import MappingNetwork
import imageio


/home/jang/anaconda3/envs/wouaf_latte/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jang/anaconda3/envs/wouaf_latte/lib/python3.12/site-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/home/jang/anaconda3/envs/wouaf_latte/lib/python3.12/site-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


In [2]:
parser = argparse.ArgumentParser()
parser.add_argument("--config", type=str, default="./configs/t2x/t2v_sample_watermark.yaml")

args = parser.parse_args(args=[])
args = OmegaConf.load(args.config)

In [3]:
torch.manual_seed(args.seed)
torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
decoding_network = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).to(device)
decoding_network.fc = torch.nn.Linear(2048, 32).to(device)
decoding_network_state_dict = torch.load(args.decoder_path) 
parameter_names = dict(zip(decoding_network_state_dict.keys(), decoding_network.state_dict().keys()))
decoding_network_state_dict = OrderedDict(dict((parameter_names[key], value) for (key, value) in decoding_network_state_dict.items()))
decoding_network.load_state_dict(decoding_network_state_dict)
decoding_network.eval()

/tmp/ipykernel_2033883/1167479797.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  decoding_network_state_dict = torch.load(args.decoder_path)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [5]:
args.decoder_path

'/home/jang/cvpr2025/WOUAF/output/num_frames_16_resolution_256_32bit/decoding_network.pth'

In [6]:
binary_string = "10100101010000111101010101111110"
binary_list = [int(bit) for bit in binary_string]
target_message = torch.tensor(binary_list).float().to(device)

In [287]:
from load_video_dataset import video_transforms, get_dataset
from torchvision import transforms

In [334]:
transform = transforms.Compose([
                    video_transforms.ToTensorVideo(),
                    # transforms.ToTensor(),
                    # transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=True)
                    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=True)
                ])

In [335]:
binary_string = "10100101010000111101010101111110"
binary_list = [int(bit) for bit in binary_string]
target_message = torch.tensor(binary_list).float().to(device)

In [347]:
dir1_list = os.listdir("/home/jang/cvpr2025/WOUAF/sample_videos_watermark")

In [358]:
# video_path = "/home/jang/cvpr2025/WOUAF/sample_videos_watermark/Yellow_and_black_tropical_fish_dart_through_the_sea._0000.mp4"
mean_list = []
video_dataset = []
for video_path in dir1_list:
    if "mp4" in video_path:
        vframes, _, _ = torchvision.io.read_video(filename=os.path.join("/home/jang/cvpr2025/WOUAF/sample_videos_watermark/", video_path), pts_unit='sec', output_format='TCHW')
        video_dataset.append(vframes)
        vframes = transform(vframes)
        decoded_message = decoding_network(vframes.to(device))
        reconstructed_keys = (torch.sigmoid(decoded_message) > 0.5).int()
        bit_acc = ((target_message == reconstructed_keys).sum(dim=1)) / reconstructed_keys.shape[1]
        # for i in bit_acc:
            # print(i.item())

        print(bit_acc.mean())
        mean_list.append(bit_acc.mean())

video_dataset = torch.stack(video_dataset)

tensor(0.9688, device='cuda:0')
tensor(0.7969, device='cuda:0')
tensor(0.8398, device='cuda:0')
tensor(0.6484, device='cuda:0')
tensor(0.7402, device='cuda:0')
tensor(0.9004, device='cuda:0')
